# COMP842 - Exercise 1: Blockchain Structure and Tamper Detection

In [1]:
import hashlib
import time
import json


def sha256_hash(data: str) -> str:
    """Generate a SHA-256 hash (hex digest) from a string."""
    return hashlib.sha256(data.encode()).hexdigest()

## Merkle Root Function

In [2]:
def compute_merkle_root(transactions):
    """
    Compute the Merkle Root of a list of transactions.
    Hashes each transaction individually, then pairs and hashes
    repeatedly until a single root hash remains.
    If there is an odd number of hashes at any level, the last
    hash is duplicated.
    """
    if not transactions:
        return sha256_hash("")

    # Hash each transaction
    tx_hashes = [sha256_hash(json.dumps(tx, sort_keys=True)) for tx in transactions]

    while len(tx_hashes) > 1:
        if len(tx_hashes) % 2 != 0:
            tx_hashes.append(tx_hashes[-1])  # duplicate last hash if odd count

        next_level = []
        for i in range(0, len(tx_hashes), 2):
            combined = tx_hashes[i] + tx_hashes[i + 1]
            next_level.append(sha256_hash(combined))

        tx_hashes = next_level

    return tx_hashes[0]

## Block Class

In [3]:
class Block:
    def __init__(self, height, previous_hash, transactions, timestamp=None):
        self.height = height
        self.timestamp = timestamp if timestamp is not None else time.time()
        self.previous_hash = previous_hash
        self.transactions = transactions
        self.merkle_root = compute_merkle_root(transactions)
        self.hash = self.compute_hash()

    def compute_hash(self):
        """Hash the block header (height, timestamp, previous_hash, merkle_root)."""
        block_header = json.dumps({
            "height": self.height,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "merkle_root": self.merkle_root
        }, sort_keys=True)
        return sha256_hash(block_header)

    def __repr__(self):
        return (
            f"Block(height={self.height}, "
            f"hash={self.hash[:12]}..., "
            f"previous_hash={self.previous_hash[:12]}..., "
            f"merkle_root={self.merkle_root[:12]}..., "
            f"transactions={self.transactions})"
        )

## Blockchain Class

In [4]:
class Blockchain:
    def __init__(self):
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_block = Block(
            height=0,
            previous_hash="0" * 64,
            transactions=["Genesis Block"],
            timestamp=0
        )
        self.chain.append(genesis_block)

    def get_latest_block(self):
        return self.chain[-1]

    def add_block(self, transactions):
        previous_block = self.get_latest_block()
        new_block = Block(
            height=previous_block.height + 1,
            previous_hash=previous_block.hash,
            transactions=transactions
        )
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self):
        """
        Validate the entire chain.
        Returns (True, None) if valid.
        Returns (False, height_of_first_invalid_block) if invalid.
        """
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # Check 1: has the block's transaction data been changed
            # without updating the merkle root?
            if current.merkle_root != compute_merkle_root(current.transactions):
                return False, current.height

            # Check 2: does the stored hash match a freshly computed hash
            # of this block's header?
            if current.hash != current.compute_hash():
                return False, current.height

            # Check 3: does this block correctly reference the previous
            # block's hash?
            if current.previous_hash != previous.hash:
                return False, current.height

        return True, None

## Build a Blockchain with 10 Blocks (Including Genesis)

In [7]:
blockchain = Blockchain()

transactions_list = [
    "Alice pays Bob 5 BTC",
    "Bob pays Charlie 2.5 BTC",
    "Charlie pays Dave 1 BTC",
    "Dave pays Eve 3.2 BTC",
    "Eve pays Frank 0.8 BTC",
    "Frank pays Grace 4.1 BTC",
    "Grace pays Heidi 2 BTC",
    "Heidi pays Alice 1.5 BTC",
    "Alice pays Charlie 6 BTC",
]

for i in range(1, 10):
    blockchain.add_block(transactions=[transactions_list[i - 1]])

#for i in range(1, 10):
    #blockchain.add_block(transactions=[f"Transaction data for block {i}"])

print("=== Blockchain BEFORE tampering ===")
for block in blockchain.chain:
    print(block)

is_valid, invalid_height = blockchain.is_chain_valid()
print(f"\nIs blockchain valid? {is_valid}")

=== Blockchain BEFORE tampering ===
Block(height=0, hash=5ab41f4e8376..., previous_hash=000000000000..., merkle_root=f106434e080b..., transactions=['Genesis Block'])
Block(height=1, hash=d2737cd77ec4..., previous_hash=5ab41f4e8376..., merkle_root=72f51e55dbe2..., transactions=['Alice pays Bob 5 BTC'])
Block(height=2, hash=648b5ebcec24..., previous_hash=d2737cd77ec4..., merkle_root=54eb6b73b120..., transactions=['Bob pays Charlie 2.5 BTC'])
Block(height=3, hash=9eae0f3666ea..., previous_hash=648b5ebcec24..., merkle_root=35ca37510520..., transactions=['Charlie pays Dave 1 BTC'])
Block(height=4, hash=960fa6817145..., previous_hash=9eae0f3666ea..., merkle_root=6fcf72d5c566..., transactions=['Dave pays Eve 3.2 BTC'])
Block(height=5, hash=ee4fe8818791..., previous_hash=960fa6817145..., merkle_root=6b0557251d30..., transactions=['Eve pays Frank 0.8 BTC'])
Block(height=6, hash=d67a18b1889d..., previous_hash=ee4fe8818791..., merkle_root=7f25eca72a80..., transactions=['Frank pays Grace 4.1 BTC']

## Task 6: Tamper with the 5th Block's Transaction Data\n\n(Genesis = 1st block -> index 0, so the 5th block is at index 4). We deliberately do **not** recalculate its hash or merkle root.

In [8]:
tampered_block = blockchain.chain[4]
tampered_block.transactions = ["TAMPERED transaction data"]

print("=== Blockchain AFTER tampering block 5 (index 4) ===")
for block in blockchain.chain:
    print(block)

is_valid, invalid_height = blockchain.is_chain_valid()
print(f"\nIs blockchain valid after tampering? {is_valid}")
if not is_valid:
    print(f"First invalid block detected at height: {invalid_height}")

=== Blockchain AFTER tampering block 5 (index 4) ===
Block(height=0, hash=5ab41f4e8376..., previous_hash=000000000000..., merkle_root=f106434e080b..., transactions=['Genesis Block'])
Block(height=1, hash=d2737cd77ec4..., previous_hash=5ab41f4e8376..., merkle_root=72f51e55dbe2..., transactions=['Alice pays Bob 5 BTC'])
Block(height=2, hash=648b5ebcec24..., previous_hash=d2737cd77ec4..., merkle_root=54eb6b73b120..., transactions=['Bob pays Charlie 2.5 BTC'])
Block(height=3, hash=9eae0f3666ea..., previous_hash=648b5ebcec24..., merkle_root=35ca37510520..., transactions=['Charlie pays Dave 1 BTC'])
Block(height=4, hash=960fa6817145..., previous_hash=9eae0f3666ea..., merkle_root=6fcf72d5c566..., transactions=['TAMPERED transaction data'])
Block(height=5, hash=ee4fe8818791..., previous_hash=960fa6817145..., merkle_root=6b0557251d30..., transactions=['Eve pays Frank 0.8 BTC'])
Block(height=6, hash=d67a18b1889d..., previous_hash=ee4fe8818791..., merkle_root=7f25eca72a80..., transactions=['Frank